# 十三式太極拳｜用 Gemini 畫正面＋背面動作圖

**手機操作只要三步：**
1. 點上方選單「執行階段 → 全部執行」（或每個格子左邊的 ▶ 依序按）。
2. 第 2 格會出現輸入框，貼上你的 **Gemini API 金鑰** 後按 Enter（金鑰只留在這個 Colab，不會存檔）。
3. 等待圖片出現。每張圖左半是**正面**、右半是**背面**。

想只畫某一式，把第 3 格的 `要畫的招式` 改成號碼，例如 `[3]` 只畫白鶴亮翅；`'全部'` 畫 13 張。

> AI 畫的手指、左右手偶爾會出錯，請對照老師示範或 YouTube 影片確認。

In [ ]:
!pip -q install -U google-genai

In [ ]:
import getpass, os
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY:
    API_KEY = getpass.getpass('請貼上 Gemini API 金鑰後按 Enter：')
from google import genai
client = genai.Client(api_key=API_KEY)
print('✅ 金鑰設定完成')

In [ ]:
# 要畫的招式：填號碼清單，例如 [3] 或 [2, 4, 8]；填 '全部' 就畫 13 張
要畫的招式 = [3]

POSES = [
 [
  "起勢",
  "Commencing",
  "standing with feet shoulder-width apart, knees slightly bent, both arms extended forward at belly height, palms facing down, shoulders relaxed",
  "兩臂上舉時肩不聳、肘下垂；下按時屈膝鬆胯，上身正直。"
 ],
 [
  "野馬分鬃",
  "Part the Wild Horse's Mane",
  "left bow stance (left knee bent above the left toes, right leg extended behind, right foot turned out 45 degrees); left hand in front at eye level, palm facing obliquely up; right hand pressing down beside the right hip, palm down; torso upright",
  "分手與弓步同時完成；前膝對準腳尖；以腰帶手。"
 ],
 [
  "白鶴亮翅",
  "White Crane Spreads Its Wings",
  "left empty stance: weight on the bent right leg, left toes lightly touching the ground in front; right hand raised beside the right forehead, palm facing inward; left hand pressing down beside the left hip, palm down",
  "虛步重心在後腿；右手上提沉肩，左手下按坐腕。"
 ],
 [
  "左摟膝拗步",
  "Brush Knee and Push",
  "left bow stance; left hand pressing down beside the left hip after brushing past the left knee, palm down; right palm pushing forward at nose height, wrist settled, fingers pointing up",
  "推掌沉肩垂肘、坐腕舒指；由腰胯帶動手臂。"
 ],
 [
  "手揮琵琶",
  "Play the Pipa",
  "weight on the bent right leg, left heel touching the ground in front with toes raised; left hand in front at nose height, palm facing right, fingers up; right hand near the inside of the left elbow, palm facing left, as if holding a pipa",
  "兩臂合抱如抱琵琶；重心坐實右腿。"
 ],
 [
  "倒卷肱",
  "Repulse the Monkey",
  "stepping backward: weight on the rear left leg, right foot in front with only light weight; right palm pushing forward at shoulder height, fingers up; left hand withdrawn to the left waist, palm up",
  "退步前腳掌先著地；推掌與收手同時到位。"
 ],
 [
  "攬雀尾（按）",
  "Grasp the Bird's Tail - Push",
  "left bow stance; both palms pushing forward at chest height, shoulder-width apart, wrists settled, fingers pointing up",
  "掤捋擠按分清；按時先沉後推，力由腿起。"
 ],
 [
  "單鞭",
  "Single Whip",
  "left bow stance facing forward-left; right hand in a hook (all five fingertips pinched together, pointing down) extended to the right-back at shoulder height; left palm pushing forward, fingers up",
  "勾手五指撮攏；推掌、弓步、轉腰同時完成。"
 ],
 [
  "雲手",
  "Cloud Hands",
  "feet parallel about shoulder-width apart, knees bent; waist turned to the left; left hand at face height with palm facing the face; right hand at belly height, palm facing down and inward",
  "以腰為軸帶動兩手畫圓；身體不起伏。"
 ],
 [
  "高探馬",
  "High Pat on Horse",
  "right empty stance setup: weight on the bent right leg, left toes touching the ground in front; right palm pushing forward at eye level; left hand at the waist, palm up",
  "虛步重心在右腿；推掌沉肩垂肘。"
 ],
 [
  "右蹬腳",
  "Right Heel Kick",
  "standing on the slightly bent left leg; right leg extended forward-right at hip height, heel pushing out, toes pulled back; both arms opened to the sides at shoulder height, palms facing out, right arm above the right leg",
  "支撐腿微屈站穩；腳跟蹬出、腳尖回勾。"
 ],
 [
  "十字手",
  "Cross Hands",
  "standing with feet parallel shoulder-width apart; arms rounded and crossed in front of the chest at the wrists, right hand on the outside, palms facing inward",
  "兩臂撐圓；起身時頭頂上領。"
 ],
 [
  "收勢",
  "Closing",
  "standing upright, feet together, arms relaxed at the sides, calm gaze forward",
  "兩手下落配合呼氣；併步靜立片刻。"
 ]
]

# 依序嘗試的圖片模型（第一個不能用就自動換下一個）
MODELS = ['gemini-3-pro-image-preview', 'gemini-2.5-flash-image', 'gemini-2.5-flash-image-preview']

In [ ]:
from IPython.display import display, Image as IPImage, Markdown
import pathlib
out = pathlib.Path('/content/taichi'); out.mkdir(exist_ok=True)

def prompt_for(name_zh, name_en, pose):
    return (f"Two-panel instructional illustration for learning Tai Chi, form '{name_en}' ({name_zh}). "
            "The SAME person in the IDENTICAL pose in both panels. LEFT panel: front view, facing the camera. "
            "RIGHT panel: back view, seen from directly behind (so the person's left hand is on the viewer's left). "
            "Subject: a middle-aged East Asian tai chi practitioner in a loose white traditional tai chi uniform and black cloth shoes, "
            "full body visible from head to feet, plain light grey studio background, soft even lighting, realistic style. "
            f"Pose: {pose}. "
            "Show correct tai chi body mechanics: relaxed sunken shoulders, elbows dropped, waist relaxed and hips settled down, "
            "spine upright, knees aligned with toes. Anatomically correct hands with five fingers, clearly visible palm orientation. "
            "No text, no labels, no watermark.")

def draw(i):
    name_zh, name_en, pose, tip = POSES[i-1]
    display(Markdown(f'### 第 {i} 式　{name_zh}\n**要領**：{tip}'))
    last = None
    for m in MODELS:
        try:
            r = client.models.generate_content(model=m, contents=prompt_for(name_zh, name_en, pose))
            for part in r.candidates[0].content.parts:
                if getattr(part, 'inline_data', None) and part.inline_data.data:
                    f = out / f'{i:02d}_{name_zh}.png'
                    f.write_bytes(part.inline_data.data)
                    display(IPImage(data=part.inline_data.data, width=640))
                    print(f'（模型：{m}，已存成 {f.name}）')
                    return
            last = '模型沒有回傳圖片'
        except Exception as e:
            last = e
    print('❌ 這一式沒畫成功：', last)

todo = range(1, 14) if 要畫的招式 == '全部' else 要畫的招式
for i in todo:
    draw(i)

## （選用）把圖片存到雲端硬碟
執行下一格會要求登入 Google 授權，圖片會存到「我的雲端硬碟 / 太極動作圖」。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
dst = pathlib.Path('/content/drive/MyDrive/太極動作圖'); dst.mkdir(exist_ok=True)
for f in out.glob('*.png'):
    shutil.copy(f, dst / f.name)
print('已存到：我的雲端硬碟/太極動作圖')